In [1]:
from elasticsearch import Elasticsearch

In [2]:
es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic","RsoKcNYcuKHgxWPPnAHj"),
    ca_certs=r"E:\elastic-stack\elasticsearch-9.3.2\config\certs\http_ca.crt"
)
es.ping()

True

# Prepare the data

In [3]:
import pandas as pd

df = pd.read_csv("myntra_products_catalog.csv").loc[:499]
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [4]:
df.isna().value_counts()

ProductID  ProductName  ProductBrand  Gender  Price (INR)  NumImages  Description  PrimaryColor
False      False        False         False   False        False      False        False           468
                                                                                   True             32
Name: count, dtype: int64

In [5]:
df.fillna("None", inplace=True)

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White
...,...,...,...,...,...,...,...,...
495,10018075,Puma Men Blue Sneakers,Puma,Men,1749,5,"A pair of round-toe blue sneakers, has regular...",Blue
496,10009687,SPYKAR Women Blue & White Striped Adora Skinny...,SPYKAR,Women,1034,7,Blue and White striped 5-pocket mid-rise jeans...,Blue
497,10017425,DKNY Unisex Black & Grey Printed Large Trolley...,DKNY,Unisex,13275,7,"Black and grey printed large trolley bag, secu...",Black
498,10017615,Parx Men Pink Slim Fit Solid Casual Shirt,Parx,Men,612,5,"Pink solid casual shirt, has a spread collar, ...",Pink


# Convert the relevant field to Vector using BERT model

In [6]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-mpnet-base-v2')

e:\elastic-stack\elasticsearch-9.3.2\semantic search\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6861.04it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
df["DescriptionVector"] = df["Description"].apply(lambda x: model.encode(x))

In [8]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor,DescriptionVector
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black,"[0.027645713, -0.0026342352, -0.00358842, 0.05..."
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige,"[-0.024660675, -0.028755445, -0.020332461, 0.0..."
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink,"[-0.04694327, 0.08182781, 0.048335183, -0.0001..."
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue,"[-0.015098754, -0.010285407, 0.009487309, -0.0..."
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White,"[-0.017746488, 0.0062096016, 0.021813974, 0.02..."


In [9]:
es.ping()

True

# Create new index in ElasticSearch!

In [10]:
# 1. Define the mapping directly to avoid ImportErrors
index_mapping_internal = {
    "properties": {
        "ProductID": {"type": "integer"},
        "ProductName": {"type": "text"},
        "ProductBrand": {"type": "text"},
        "Gender": {"type": "text"},
        "Price (INR)": {"type": "long"},
        "Description": {"type": "text"},
        "PrimaryColor": {"type": "text"},
        "DescriptionVector": {
            "type": "dense_vector",
            "dims": 768, # Matches your all-mpnet-base-v2 model
            "index": True,
            "similarity": "l2_norm"
        }
    }
}

# 2. Delete the index if it exists so the 'create' command can run fresh
if es.indices.exists(index="all_products"):
    es.indices.delete(index="all_products")

# 3. This line will now give you the exact Output you want
response = es.indices.create(index="all_products", mappings=index_mapping_internal)
response

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'all_products'})

# Ingest the data into index

In [11]:
record_list = df.to_dict("records")

In [12]:
for record in record_list:
    try:
        es.index(index="all_products", document=record, id=record["ProductID"])
    except Exception as e:
        print(e)

In [13]:
es.count(index="all_products")

ObjectApiResponse({'count': 425, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

# Search the data

In [17]:
input_keyword = "Blue Shoes"
vector_of_input_keyword = model.encode(input_keyword)

query = {
    "field" : "DescriptionVector",
    "query_vector" : vector_of_input_keyword,
    "k" : 2,
    "num_candidates" : 500, 
}

res = es.search(index="all_products", knn=query, source=["ProductName", "Description"])
res["hits"]["hits"]

[{'_index': 'all_products',
  '_id': '10018013',
  '_score': 0.6142942,
  '_source': {'ProductName': 'Puma Men Blue Sneakers',
   'Description': 'A pair of round-toe blue sneakers, has regular styling, lace-up detailTextile upperCushioned footbedTextured and patterned outsoleWarranty: 3 monthsWarranty provided by brand/manufacturer'}},
 {'_index': 'all_products',
  '_id': '10018075',
  '_score': 0.6142942,
  '_source': {'ProductName': 'Puma Men Blue Sneakers',
   'Description': 'A pair of round-toe blue sneakers, has regular styling, lace-up detailTextile upperCushioned footbedTextured and patterned outsoleWarranty: 3 monthsWarranty provided by brand/manufacturer'}}]